In [129]:
from langgraph.graph import StateGraph
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from typing import TypedDict
from langgraph.constants import END, START
from pydantic import BaseModel, Field
from typing import Annotated, Literal
import operator

In [130]:
load_dotenv()

True

In [131]:
model = ChatOllama(
    model = 'qwen3:1.7b'
)

In [132]:
class Sentiment(BaseModel):

    sentiment: Literal['Positive', 'Negative'] = Field(description="Sentiment of the review")

In [133]:
structured_model = model.with_structured_output(Sentiment)

In [134]:
class analysis(TypedDict):

    review: str
    find_sentiment: Literal["Positive", "Negative"]
    positive_response: str
    negative_response: str
    result: str

In [135]:
def review(state: analysis):

    review = state['review']
    return {'review': review}

def find_sentiment(state: analysis):

    prompt = f"Analyse deeply and find out the sentiment of the review {state['review']}"
    result = structured_model.invoke(prompt)

    return {'find_sentiment': result.sentiment}

def positive_response(state: analysis):

    prompt = f"Generate a reply for the positive review {state['review']}"
    result = model.invoke(prompt)

    return {'positive_response': result.content}

def negative_response(state: analysis):

    prompt = f"Analyse the type of issue, urgency, tone in the review and Generate a reply for the negative review {state['review']}"
    result = model.invoke(prompt)

    return {'negative_response': result.content}

def check_condition(state: analysis) -> Literal['positive_response', 'negative_response']:

    if state['find_sentiment'] == 'Positive':
        return "positive_response"
    else:
        return "negative_response"
        


In [136]:
graph = StateGraph(analysis)

graph.add_node("find_sentiment", find_sentiment)
graph.add_node("review", review)
graph.add_node("positive_response", positive_response)
graph.add_node("negative_response", negative_response)

graph.add_edge(START, "review")
graph.add_edge("review", "find_sentiment")
graph.add_conditional_edges("find_sentiment", check_condition)

graph.add_edge("positive_response", END)
graph.add_edge("negative_response", END)

workflow = graph.compile()

In [137]:
workflow.invoke({
    'review': "I’m disappointed with how things turned out. I expected better results, and there’s definitely room for improvement"
})

{'review': 'I’m disappointed with how things turned out. I expected better results, and there’s definitely room for improvement',
 'find_sentiment': 'Negative',
 'negative_response': '**Type of Issue:**  \nThe issue appears to be related to **product/service quality** or **customer experience**, likely involving a specific product, service, or process that did not meet expectations. The user is expressing dissatisfaction with the outcome and highlighting areas for improvement.\n\n**Urgency:**  \n**Moderate**. The issue is not time-sensitive, but the user is clearly frustrated and wants a resolution.\n\n**Tone:**  \n**Negative** but **empathetic**. The reviewer expresses disappointment without being overly critical, focusing on the need for improvement.\n\n---\n\n**Reply:**  \nThank you for sharing your feedback. I’m sorry to hear that things didn’t meet your expectations. We truly value your input and are committed to improving our service. If you’d like, please let us know how we can 